In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline
import re
import json
import torch

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [2]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### Chat Template

In [4]:
print(tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba C

In [5]:
messages = [
    {"user": "who are you?"}
]

In [6]:
print(tokenizer.apply_chat_template(messages, tokenize=False))

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>



In [7]:
tools = [
    {
        "name": "get_weather",
        "description": "Get weather for a city",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string"
                }
            },
            "required": ["city"]
        }
    }
]
messages = [
    {"role": "user", "content": "what is the weather in Jakarta?"}
]

In [12]:
text = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    # add_generation_prompt=True
)

In [13]:
text

'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>\n{"name": "get_weather", "description": "Get weather for a city", "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{"name": <function-name>, "arguments": <args-json-object>}\n</tool_call><|im_end|>\n<|im_start|>user\nwhat is the weather in Jakarta?<|im_end|>\n'

In [14]:
print(text)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"name": "get_weather", "description": "Get weather for a city", "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
what is the weather in Jakarta?<|im_end|>



In [22]:
import re
import json
from typing import Optional, Dict, Any

def parse_tool_call(text: str) -> Optional[Dict[str, Any]]:
    pattern = r"<tool_call>\s*(\{.*?\})\s*</tool_call><|im_end|>"
    match = re.search(pattern, text, re.DOTALL)
    if not match:
        return None

    try:
        return json.loads(match.group(1))
    except json.JSONDecodeError:
        return None    

In [20]:
text = '<tool_call>{"name": "get_weather", "arguments": {"city": "Jakarta"}}</tool_call><|im_end|>'
result = parse_tool_call(text)

In [21]:
result

{'name': 'get_weather', 'arguments': {'city': 'Jakarta'}}

In [23]:
messages = [
    {"role": "user", "content": "what is the weather in Jakarta?"},
    {
        "role": "assistant",
        "content": "",
        "tool_calls": [
            {
                "name": "get_weather",
                "arguments": {"city": "Jakarta"}
            }
        ]
    },
    {
        "role": "tool",
        "content": "The weather in Jakarta is 30°C and sunny."
    }
]

In [26]:
print(
    tokenizer.apply_chat_template(
        messages, 
        tokenize=False,
        tools=tools,
    )
)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"name": "get_weather", "description": "Get weather for a city", "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
what is the weather in Jakarta?<|im_end|>
<|im_start|>assistant
<tool_call>
{"name": "get_weather", "arguments": {"city": "Jakarta"}}
</tool_call><|im_end|>
<|im_start|>user
<tool_response>
The weather in Jakarta is 30°C and sunny.
</tool_response><|im_end|>



In [27]:
def get_last_assistant(text: str):
    pattern = r"<\|im_start\|>assistant\s*(.*?)<\|im_end\|>"
    matches = re.findall(pattern, text, flags=re.DOTALL)

    if matches:
        return matches[-1].strip()

    fallback_pattern = r"<\|im_start\|>assistant\s*(.*)$"
    fallback_matches = re.findall(fallback_pattern, text, flags=re.DOTALL)

    if fallback_matches:
        return fallback_matches[-1].strip()

    return text

In [ ]:
def clean(text):
    for tok in special_tokens:
        text = text.replace(tok, "")
    return text.strip()

In [28]:
decoded = '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nGive three tips for staying healthy.<|im_end|>\n<|im_start|>assistant\nCertainly! Here are three effective tips for maintaining good health:\n\n1. **Maintain a Balanced Diet**: Eating a variety of nutritious foods is crucial for overall health. Focus on incorporating plenty of fruits, vegetables, whole grains, and lean proteins into your diet. Limit the intake of processed foods, sugars, and unhealthy fats. Staying hydrated is also important, so make sure to drink enough water throughout the day.\n\n2. **Regular Exercise**: Aim for at least 150 minutes of moderate aerobic activity or 75 minutes of vigorous activity each week, along with muscle-strengthening exercises on two or more days a week'

In [29]:
get_last_assistant(decoded)

'Certainly! Here are three effective tips for maintaining good health:\n\n1. **Maintain a Balanced Diet**: Eating a variety of nutritious foods is crucial for overall health. Focus on incorporating plenty of fruits, vegetables, whole grains, and lean proteins into your diet. Limit the intake of processed foods, sugars, and unhealthy fats. Staying hydrated is also important, so make sure to drink enough water throughout the day.\n\n2. **Regular Exercise**: Aim for at least 150 minutes of moderate aerobic activity or 75 minutes of vigorous activity each week, along with muscle-strengthening exercises on two or more days a week'

In [32]:
decoded_with_end = f"{decoded}<|im_end|>"

In [33]:
print(decoded_with_end)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Give three tips for staying healthy.<|im_end|>
<|im_start|>assistant
Certainly! Here are three effective tips for maintaining good health:

1. **Maintain a Balanced Diet**: Eating a variety of nutritious foods is crucial for overall health. Focus on incorporating plenty of fruits, vegetables, whole grains, and lean proteins into your diet. Limit the intake of processed foods, sugars, and unhealthy fats. Staying hydrated is also important, so make sure to drink enough water throughout the day.

2. **Regular Exercise**: Aim for at least 150 minutes of moderate aerobic activity or 75 minutes of vigorous activity each week, along with muscle-strengthening exercises on two or more days a week<|im_end|>


In [36]:
get_last_assistant(decoded)

'Certainly! Here are three effective tips for maintaining good health:\n\n1. **Maintain a Balanced Diet**: Eating a variety of nutritious foods is crucial for overall health. Focus on incorporating plenty of fruits, vegetables, whole grains, and lean proteins into your diet. Limit the intake of processed foods, sugars, and unhealthy fats. Staying hydrated is also important, so make sure to drink enough water throughout the day.\n\n2. **Regular Exercise**: Aim for at least 150 minutes of moderate aerobic activity or 75 minutes of vigorous activity each week, along with muscle-strengthening exercises on two or more days a week'